# 01. ESM-2 임베딩 + 데이터 준비
ESMFold는 openfold 의존성 문제로 Colab에서 사용 불가. ESM-2 임베딩만 계산하고 바로 Phase 2로 진행.

In [ ]:
# 환경 설정
import os, sys
os.chdir('/content')
if not os.path.exists('brain_idp_flow'):
    !git clone https://github.com/layeredlabs06/brain-idp-flow.git brain_idp_flow
os.chdir('brain_idp_flow')
!pip install -e . -q
!pip install fair-esm -q
sys.path.insert(0, '/content/brain_idp_flow/src')

import numpy as np
import torch
from brain_idp_flow.targets import load_targets

targets = load_targets('configs/targets.yaml')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# 데이터셋 빌드 (PED fallback 사용)
!python scripts/build_dataset.py --out data/train.npz --max-len 160

In [ ]:
# ESM-2 임베딩 사전 계산 (openfold 불필요)
from brain_idp_flow.features.seq_embed import ESM2Embedder

embedder = ESM2Embedder(device=device)
seq_embeddings = {}
sid = 0

for tid, target in targets.items():
    # WT
    emb = embedder.embed_single(target.sequence)
    seq_embeddings[sid] = emb.cpu()
    print(f'{tid} WT: seq_id={sid}, emb={emb.shape}')
    sid += 1
    # Mutants
    for mut in target.mutations:
        mut_seq = target.mutant_sequence(mut)
        emb = embedder.embed_single(mut_seq)
        seq_embeddings[sid] = emb.cpu()
        print(f'  {mut.id}: seq_id={sid}')
        sid += 1

torch.save(seq_embeddings, 'data/seq_embeddings.pt')
print(f'\nTotal: {len(seq_embeddings)} embeddings saved')

In [ ]:
# 확인
import os
print(f'Dataset: {os.path.getsize("data/train.npz") / 1e6:.1f} MB')
print(f'Embeddings: {len(seq_embeddings)} sequences')
print('\n다음: 02_train_flow.ipynb 에서 학습 시작')